# TFScope: protein DBD → PWM → consensus → logo

Given a **DNA-binding-domain (DBD) protein sequence**, this notebook uses the canonical **v24** model (`contact_v24_seed42`) to predict the binding-motif PWM, print the consensus sequence, and draw the sequence logo.

**Important:** v24 is trained on **DBD crops** (the domain only, ~40–170 aa), not full-length proteins. Feed the DBD region — full-length input is out-of-distribution. See the MyoD1 bHLH example below.

In [ ]:
import os, sys, json
os.environ.setdefault('TORCH_HOME', '/data1/leihuang/.cache/torch')
os.environ.setdefault('TRANSFORMERS_OFFLINE', '1')
sys.path.insert(0, '../src'); sys.path.insert(0, 'src')
import numpy as np, torch, torch.nn.functional as F
import matplotlib.pyplot as plt, logomaker, pandas as pd
from tfscope.config import TFScopeConfig
from tfscope.models.tfscope import TFScopeModel
from tfscope.data.dataset import AA_TO_TOKEN

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CKPT = '/data1/leihuang/project/TFScope/checkpoints/v24_contact/contact_v24_seed42'
BASES = np.array(list('ACGT'))
print('device:', DEVICE)

In [ ]:
# ---- load the v24 model once ----
cfg = TFScopeConfig()
for k, v in json.load(open(f'{CKPT}/config.json')).items():
    if hasattr(cfg, k):
        try: setattr(cfg, k, type(getattr(cfg, k))(v))
        except Exception: setattr(cfg, k, v)
cfg.use_retrieval = False
model = TFScopeModel(cfg).to(DEVICE).eval()
sd = torch.load(f'{CKPT}/ckpt_best.pt', map_location=DEVICE, weights_only=False)
model.load_state_dict(sd.get('model', sd), strict=False)
print('v24 loaded (frozen ESM-2 650M + LoRA16, residue-MoE, v18 contact head, span gate)')

In [ ]:
@torch.no_grad()
def predict_pwm(seq, family_id=0):
    """DBD protein sequence -> (pwm_core 4xL, consensus str). Uses the model's
    span gate to crop to the predicted motif; the whole input is treated as DBD."""
    seq = seq.strip().upper()
    tok = torch.tensor([[AA_TO_TOKEN.get(a, 4) for a in seq]], dtype=torch.long, device=DEVICE)
    dbd = torch.ones(1, len(seq), dtype=torch.bool, device=DEVICE)   # whole crop = DBD
    fid = torch.tensor([int(family_id)], device=DEVICE)
    gl, pl, _ = model(tok, dbd, fid, retrieved_pwms=None, retrieved_masks=None,
                      retrieved_sims=None, recog_prior=None)
    gate = gl.sigmoid()[0].cpu().numpy()
    pwm = F.softmax(pl, 1)[0].cpu().numpy()             # (4, 42) full
    L = max(4, int((gate > 0.5).sum()))                 # predicted motif length
    core = pwm[:, :L]                                    # (4, L) motif core
    consensus = ''.join(BASES[core.argmax(0)])
    return core, consensus

def plot_logo(core, title='TFScope predicted motif'):
    """core: (4, L) probability matrix -> information-content logo."""
    P = np.clip(core.T, 1e-9, 1.0)                       # (L, 4)
    ic = (P * np.log2(P / 0.25)).sum(1, keepdims=True)   # bits per position
    df = pd.DataFrame(P * ic, columns=list('ACGT'))
    fig, ax = plt.subplots(figsize=(max(3, 0.6 * len(df)), 2.2))
    logomaker.Logo(df, ax=ax, color_scheme='classic')
    ax.set_ylabel('bits'); ax.set_ylim(0, 2); ax.set_title(title)
    ax.set_xticks(range(len(df))); ax.set_xticklabels(range(1, len(df) + 1))
    plt.tight_layout(); plt.show()
print('helpers ready: predict_pwm(seq, family_id), plot_logo(core)')

## Example: MyoD1 bHLH DBD
The muscle-regulator MyoD1 basic-helix-loop-helix domain binds the E-box `CANNTG` (muscle E-box `CACCTG`/`CAGCTG`). `family_id=3` = bHLH.

In [ ]:
MYOD1_DBD = 'RKAATMRERRRLSKVNEAFETLKRCTSSNPNQRLPKVEILRNAIRYIEGLQA'  # bHLH DBD crop
core, cons = predict_pwm(MYOD1_DBD, family_id=3)
print('consensus:', cons, '  (motif length', core.shape[1], 'bp)')
plot_logo(core, title=f'MyoD1 bHLH — predicted motif ({cons})')

## Your own protein
Paste a **DBD-cropped** sequence below. Family ids (v24 10-family scheme): 0=Other, 1=?, 3=bHLH, 4=Homeodomain/POU, ... — the family head is near-inert in v24, so `family_id` barely changes the prediction; leave it at a sensible family or 0.

In [ ]:
MY_SEQ = 'RKAATMRERRRLSKVNEAFETLKRCTSSNPNQRLPKVEILRNAIRYIEGLQA'  # <-- replace with your DBD
MY_FAMILY = 3
core, cons = predict_pwm(MY_SEQ, family_id=MY_FAMILY)
print('consensus:', cons, '  length', core.shape[1])
print('PWM (rows A,C,G,T):')
print(np.round(core, 3))
plot_logo(core, title=f'Predicted motif ({cons})')